In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.getOrCreate()
customers = spark.createDataFrame([
    (1,"John ","Hyderabad"),
    (2,"Alice","Chennai"),
    (3,None,"Bangalore")
],["customer_id","name","city"])
cars = spark.createDataFrame([
    (101,"Toyota","Camry",30000),
    (102,"Honda","Civic",-20000),
    (103,"Hyundai","i20",15000)
],["car_id","brand","model","price"])
sales = spark.createDataFrame([
    (1,1,101,"2024-01-01",1),
    (2,2,102,"2024-01-02",2),
    (3,99,103,"2024-01-03",1)
],["sale_id","customer_id","car_id","sale_date","quantity"])
sales = sales.withColumn("sale_date", to_date(col("sale_date")))
df = sales.join(customers,"customer_id").join(cars,"car_id")
df.groupBy("customer_id").sum("price").show()

+-----------+----------+
|customer_id|sum(price)|
+-----------+----------+
|          1|     30000|
|          2|    -20000|
+-----------+----------+



In [0]:
c_clean=customers.fillna({"name":"unknown"})
cars_clean=cars.withColumn(
    "price",
    when(col("price")<0,0).otherwise(col("price"))
)
c.display()
cars_clean.display()
sales.display()

customer_id,name,city
1,John,Hyderabad
2,Alice,Chennai
3,unknown,Bangalore


car_id,brand,model,price
101,Toyota,Camry,30000
102,Honda,Civic,0
103,Hyundai,i20,15000


sale_id,customer_id,car_id,sale_date,quantity
1,1,101,2024-01-01,1
2,2,102,2024-01-02,2
3,99,103,2024-01-03,1


In [0]:
invalid_sales=sales.join(c,"customer_id","left_anti")
sales_clean = sales.join(c, "customer_id", "inner")
invalid_sales.display()

customer_id,sale_id,car_id,sale_date,quantity
99,3,103,2024-01-03,1


In [0]:
df=sales_clean.join(c,"customer_id").join(cars_clean,"car_id")
df.show()

+------+-----------+-------+----------+--------+-----+---------+-----+---------+------+-----+-----+
|car_id|customer_id|sale_id| sale_date|quantity| name|     city| name|     city| brand|model|price|
+------+-----------+-------+----------+--------+-----+---------+-----+---------+------+-----+-----+
|   101|          1|      1|2024-01-01|       1|John |Hyderabad|John |Hyderabad|Toyota|Camry|30000|
|   102|          2|      2|2024-01-02|       2|Alice|  Chennai|Alice|  Chennai| Honda|Civic|    0|
+------+-----------+-------+----------+--------+-----+---------+-----+---------+------+-----+-----+



In [0]:
df = sales_clean.join(c, "customer_id").join(cars_clean, "car_id")
df.display()

car_id,customer_id,sale_id,sale_date,quantity,name,city,name,city,brand,model,price
101,1,1,2024-01-01,1,John,Hyderabad,John,Hyderabad,Toyota,Camry,30000
102,2,2,2024-01-02,2,Alice,Chennai,Alice,Chennai,Honda,Civic,0


In [0]:
df = df.withColumn("revenue", col("price") * col("quantity"))
df.display()

car_id,customer_id,sale_id,sale_date,quantity,name,city,brand,model,price,revenue
101,1,1,2024-01-01,1,John,Hyderabad,Toyota,Camry,30000,30000
102,2,2,2024-01-02,2,Alice,Chennai,Honda,Civic,20000,40000


In [0]:
revenue_per_customer = df.groupBy("customer_id").agg(sum("revenue").alias("total_revenue"))
revenue_per_customer.display()

customer_id,total_revenue
1,30000
2,40000


In [0]:
cars_per_brand = cars_clean.groupBy("brand").count()
cars_per_brand.display()

brand,count
Toyota,1
Honda,1
Hyundai,1


In [0]:
city_revenue = df.groupBy("city").agg(sum("revenue").alias("city_revenue"))
city_revenue.display()

city,city_revenue
Hyderabad,30000
Chennai,40000


In [0]:
df.createOrReplaceTempView("sales_data")

In [0]:
sales_clean.columns

['customer_id', 'sale_id', 'car_id', 'sale_date', 'quantity']

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Step 1: Aggregate (same as GROUP BY)
df_agg = df.groupBy("city", "customer_id") \
                   .agg(sum("revenue").alias("total"))

# Step 2: Window function (RANK)
window = Window.partitionBy("city").orderBy(col("total").desc())

df_ranked = df_agg.withColumn(
    "rnk",
    rank().over(window)
)

# Step 3: Filter top 3
result = df_ranked.filter(col("rnk") <= 3)

result.display()

city,customer_id,total,rnk
Chennai,2,40000,1
Hyderabad,1,30000,1


In [0]:
result = df.groupBy("customer_id") \
                   .agg(count("*").alias("purchases")) \
                   .filter(col("purchases") > 1)

result.display()

customer_id,purchases


In [0]:
result = df.withColumn(
    "month",
    date_format("sale_date", "yyyy-MM")
).groupBy("month") \
 .agg(sum("revenue").alias("total_revenue")) \
 .orderBy("month")

result.display()

month,total_revenue
2024-01,70000
